In [1]:
!pip install --upgrade pip setuptools wheel -q
!pip install --upgrade cmake -q
!pip install scs --prefer-binary -q
!pip install cvxpy --prefer-binary -q
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary
!pip install catboost


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Program Files\Python314\python.exe -m pip install --upgrade pip setuptools wheel -q

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from optbinning import BinningProcess
import shutil
from warnings import simplefilter
simplefilter(action = "ignore") #, category = FutureWarning

pd.set_option('display.max_rows', 500)
from sklearn.preprocessing import LabelEncoder

(CVXPY) Aug 21 06:41:49 AM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Aug 21 06:41:49 AM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple
#import plotly.graph_objects as go
#from plotly.subplots import make_subplots
import os
#import plotly.express as px

pd.set_option('display.float_format', '{:.2f}'.format)

In [4]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")

✓ Credenciales cargadas desde: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()


In [5]:
%%time
query = """

WITH pd AS (
    SELECT DISTINCT
        -- Identificadores
        a.key_value,
        a.cod_cli,
        date_format(
            date_parse(CAST(a.cod_mes AS varchar), '%Y%m') - interval '1' month,
            '%Y%m'
        ) AS codmes_lag1,
        CAST(a.cod_mes AS INTEGER) AS cod_mes,

        -- Fechas
        TRY_CAST(a.fec_constitucion AS DATE) AS fec_constitucion,

        -- Monetarios
        TRY_CAST(a.mto_pas_soles AS DOUBLE) AS mto_pas_soles,
        TRY_CAST(a.imp_trx_abonosefect_6m AS DOUBLE) AS imp_trx_abonosefect_6m,
        TRY_CAST(a.imp_trx_cargosefe_6m AS DOUBLE) AS imp_trx_cargosefe_6m,
        TRY_CAST(a.avg_trx_cargostot_3m AS DOUBLE) AS avg_trx_cargostot_3m,
    TRY_CAST(a.max_trx_abonos_3m AS DOUBLE) AS max_trx_abonos_3m,

        -- Cantidades
        TRY_CAST(a.cnt_trx_cargostot_3m AS INTEGER) AS cnt_trx_cargostot_3m,

        -- Promedios / ratios
        TRY_CAST(a.cnt_trx_abonospromtot_3m AS DOUBLE) AS cnt_trx_abonospromtot_3m,
        TRY_CAST(a.rat_trx_abonosefectot_1m AS DOUBLE) AS rat_trx_abonosefectot_1m,
        TRY_CAST(a.rat_trx_abonosefectot_3m AS DOUBLE) AS rat_trx_abonosefectot_3m,
        TRY_CAST(a.rat_trx_abonosefectot_9m AS DOUBLE) AS rat_trx_abonosefectot_9m,
        TRY_CAST(a.rat_mntcrgsefetot_1m AS DOUBLE) AS rat_mntcrgsefetot_1m,

        -- Demográficas / antigüedad
        TRY_CAST(a.num_edad_constitucion AS INTEGER) AS num_edad_constitucion,
    TRY_CAST(a.num_antiguedad AS INTEGER) AS num_antiguedad,

        -- Riesgo
        TRY_CAST(a.desc_nivel_rsg_lsb_tot AS DOUBLE) AS desc_nivel_rsg_lsb_tot,

        -- Actividad mensual
        TRY_CAST(a.cnt_meses_siningresos_12m AS INTEGER) AS cnt_meses_siningresos_12m,
        TRY_CAST(a.cnt_meses_sinegresos_12m AS INTEGER) AS cnt_meses_sinegresos_12m,

        -- Ubicación / segmentación
        a.desc_provincia,
        a.desc_departamento,
        a.cod_ubigeo_cd,
        a.cod_sectorista_id,
        a.cod_ciiu_v4,

        -- Flags (string/bool → 0/1)
        CAST(a.flg_casos_hist AS INTEGER) AS flg_casos_hist,
        CAST(a.flg_vrcn_abonos_5m_1m AS INTEGER) AS flg_vrcn_abonos_5m_1m,
        CAST(a.flg_vrcn_efe_cargos_5m_1m AS INTEGER) AS flg_vrcn_efe_cargos_5m_1m,

        -- Conteos
        a.cnt_ro_debajo_umbral,
        -- Perfil económico
        a.mto_fact_declarado_sunat,
        TRY_CAST(a.avg_cp_men_ing_12m AS DOUBLE) AS avg_cp_men_ing_12m,
        TRY_CAST(a.avg_cpmenegr_12m AS DOUBLE) AS avg_cpmenegr_12m,
        TRY_CAST(a.max_mto_cpmening_12m AS DOUBLE) AS max_mto_cpmening_12m,
        TRY_CAST(a.max_mto_cpegrmen_12m AS DOUBLE) AS max_mto_cpegrmen_12m,

        -- Exterior
        a.flg_al_ext_12m,
        a.flg_del_ext_12m,
        TRY_CAST(a.cnt_trx_sinenv_alext_12m AS INTEGER) AS cnt_trx_sinenv_alext_12m,
        TRY_CAST(a.cnt_trx_al_ext_1000_12m AS INTEGER) AS cnt_trx_al_ext_1000_12m,
        TRY_CAST(a.mto_al_ext_12m AS DOUBLE) AS mto_al_ext_12m,
        TRY_CAST(a.mto_del_ext_12m AS DOUBLE) AS mto_del_ext_12m,

        -- Reputacional / antecedentes
        a.flg_pep,
        a.cod_rsg_pep,
        a.flg_activo_pep,
        TRY_CAST(a.cnt_noticias AS INTEGER) AS cnt_noticias,
        a.flg_ros_12m,
        a.flg_alerta_12m,
        TRY_CAST(a.cnt_alerta_hist AS INTEGER) AS cnt_alerta_hist,
        TRY_CAST(a.cnt_ros_hist AS INTEGER) AS cnt_ros_hist,

        -- KYC
        a.flg_kyc_12m,
        a.flg_kyc_hist,
        TRY_CAST(a.cnt_kyc_hist AS INTEGER) AS cnt_kyc_hist,
        -- ======================================================
        -- 🔹 ACELERACIÓN / CAMBIO DE COMPORTAMIENTO
        -- ======================================================
        TRY_CAST(a.imp_trx_abonostot_1m AS DOUBLE)
            / NULLIF(TRY_CAST(a.avg_trx_abonostot_6m AS DOUBLE), 0)
            AS rat_abonos_1m_vs_6m,

        TRY_CAST(a.imp_trx_cargostot_1m AS DOUBLE)
            / NULLIF(TRY_CAST(a.avg_trx_cargostot_6m AS DOUBLE), 0)
            AS ratio_cargos_1m_vs_6m,


        -- ======================================================
        -- 🔹 CONCENTRACIÓN EN CONTRAPARTE
        -- ======================================================
        TRY_CAST(a.avg_cpmenegr_12m AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_cargostot_6m AS DOUBLE), 0)
            AS share_cp_egresos,

        TRY_CAST(a.avg_cp_men_ing_12m AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_abonostot_6m AS DOUBLE), 0)
            AS share_cp_ingresos,


        -- ======================================================
        -- 🔹 NORMALIZACIÓN DE RIESGO
        -- ======================================================
        TRY_CAST(a.cnt_ros_hist AS DOUBLE)
            / NULLIF(TRY_CAST(a.cnt_trx_cargostot_3m AS DOUBLE), 0)
            AS rat_cntros_x_cnttrxegr_3m,

        TRY_CAST(a.cnt_alerta_hist AS DOUBLE)
            / NULLIF(TRY_CAST(a.num_antiguedad AS DOUBLE), 0)
            AS alertas_por_antiguedad,


        -- ======================================================
        -- 🔹 COHERENCIA ECONÓMICA
        -- ======================================================
        TRY_CAST(a.imp_trx_abonostot_6m AS DOUBLE)
            / NULLIF(TRY_CAST(a.mto_fact_declarado_sunat AS DOUBLE), 0)
            AS rat_ing_tot_x_factura_6m,

        TRY_CAST(a.mto_pas_soles AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_abonostot_6m AS DOUBLE), 0)
            AS rat_pastot_x_ingtot_6m,



        -- ======================================================
        -- 🔹 EXPOSICIÓN AL EXTERIOR (PROPORCIONES)
        -- ======================================================
        TRY_CAST(a.mto_al_ext_12m AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_abonostot_12m AS DOUBLE), 0)
            AS ratio_egresos_exterior,

        TRY_CAST(a.mto_del_ext_12m AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_abonostot_12m AS DOUBLE), 0)
            AS rat_ing_ext_x_ing_tot_12m,

        -- ======================================================
        -- 🔹 COHERENCIA PEP / LSB
        -- ======================================================
        TRY_CAST(a.cod_rsg_pep AS DOUBLE)
            - TRY_CAST(a.desc_nivel_rsg_lsb_tot AS DOUBLE)
            AS gap_riesgo_pep_lsb

    FROM e_perm_aws.t_agg_alertas_plaft a
        WHERE a.cod_mes BETWEEN '202605' AND '202607'
            AND a.desc_subsegmento = 'BPE'
),

target AS (
    SELECT 
        codunico,
        periodo_alerta,
        tipo_alerta_n2,
        trx_riesgo_cliente,
        MAX(calificacion_monitoreo) AS flg_alerta
    FROM e_perm_aws.t_alertas_plaft
    GROUP BY codunico, periodo_alerta, tipo_alerta_n2,trx_riesgo_cliente
)

SELECT 
    a.*,
    b.tipo_alerta_n2,
    b.trx_riesgo_cliente, 
    CASE 
        WHEN b.flg_alerta = '1' THEN 1 
        ELSE 0 
    END AS target_m
FROM pd a
LEFT JOIN target b
    ON a.cod_cli = b.codunico
    AND cast(a.cod_mes as varchar) = b.periodo_alerta
--   AND codmes_lag1 = c.periodo_alerta
;"""
df = athena_query(query, database='disc_comercial')
df.head()

CPU times: total: 10.5 s
Wall time: 1min 20s


,key_value,cod_cli,codmes_lag1,cod_mes,fec_constitucion,mto_pas_soles,imp_trx_abonosefect_6m,imp_trx_cargosefe_6m,avg_trx_cargostot_3m,max_trx_abonos_3m,...,rat_cntros_x_cnttrxegr_3m,alertas_por_antiguedad,rat_ing_tot_x_factura_6m,rat_pastot_x_ingtot_6m,ratio_egresos_exterior,rat_ing_ext_x_ing_tot_12m,gap_riesgo_pep_lsb,tipo_alerta_n2,trx_riesgo_cliente,target_m
0,5FC20C5D0C19B187AAAD242986898510AD5B03D2A75DEA...,0020981707,202605,202606,2011-03-30,0.00,0.00,0.00,0.00,0.00,...,NaN,NaN,0.00,NaN,NaN,NaN,NaN,<NA>,<NA>,0
1,0682DA8030BDDC346E9F14826A323DAEB1B6B313B5F784...,0014834336,202606,202607,2016-09-16,0.00,0.00,0.00,0.00,0.00,...,NaN,NaN,0.05,0.00,0.00,0.00,NaN,<NA>,<NA>,0
2,778D6D69E6D7BE755FD9022085F14E9FC651B892B93A04...,0013076463,202604,202605,2001-08-07,20683.73,13751.80,14800.00,0.67,827828.81,...,0.00,0.08,0.28,0.02,0.00,0.00,NaN,<NA>,<NA>,0
3,6EB7D03B127910B5FBDA93481A055B12F128CE7025C476...,0018760243,202606,202607,2023-02-14,25.00,0.00,0.00,0.00,0.00,...,NaN,NaN,NaN,0.25,0.00,0.00,NaN,<NA>,<NA>,0
4,0331318F473AAE318596BB84B95B0C194B786C96E0A256...,0021907687,202605,202606,2023-09-18,0.00,0.00,0.00,0.00,1380.00,...,NaN,NaN,NaN,0.00,0.00,0.00,NaN,<NA>,<NA>,0


In [6]:

# ============================================================
# 🔍 VALIDACIÓN DE VARIABLES POST-QUERY
# ============================================================
print("=" * 80)
print("📊 RESUMEN DE DATAFRAME DEL QUERY SQL")
print("=" * 80)

print(f"\n✓ Registros totales: {len(df):,}")
print(f"✓ Columnas totales: {len(df.columns)}")

print("\n📋 LISTADO DE COLUMNAS:")
print("-" * 80)
for i, col in enumerate(df.columns, 1):
    dtype = df[col].dtype
    non_null = df[col].notna().sum()
    null_pct = (df[col].isna().sum() / len(df) * 100)
    print(f"{i:3d}. {col:40s} | Tipo: {str(dtype):20s} | Nulos: {null_pct:5.2f}%")

print("\n" + "=" * 80)
print("📌 TIPOS DE DATOS DETECTADOS:")
print("=" * 80)
print(df.dtypes)

print("\n" + "=" * 80)
print("🔗 VALIDACIÓN DE KEY VARIABLES:")
print("=" * 80)
key_vars = ['key_value', 'cod_cli', 'cod_mes', 'target_m', 'tipo_alerta_n2', 
            'mto_pas_soles', 'cnt_trx_cargostot_3m', 'num_antiguedad']
for var in key_vars:
    if var in df.columns:
        print(f"✓ {var:30s} está presente")
    else:
        print(f"✗ {var:30s} FALTA")

print("\nℹ️ Primeras 5 filas:")
print(df.head())


📊 RESUMEN DE DATAFRAME DEL QUERY SQL

✓ Registros totales: 534,404
✓ Columnas totales: 66

📋 LISTADO DE COLUMNAS:
--------------------------------------------------------------------------------
  1. key_value                                | Tipo: string               | Nulos:  0.00%
  2. cod_cli                                  | Tipo: string               | Nulos:  0.00%
  3. codmes_lag1                              | Tipo: string               | Nulos:  0.00%
  4. cod_mes                                  | Tipo: Int32                | Nulos:  0.00%
  5. fec_constitucion                         | Tipo: object               | Nulos:  6.11%
  6. mto_pas_soles                            | Tipo: float64              | Nulos:  0.00%
  7. imp_trx_abonosefect_6m                   | Tipo: float64              | Nulos:  0.00%
  8. imp_trx_cargosefe_6m                     | Tipo: float64              | Nulos:  0.00%
  9. avg_trx_cargostot_3m                     | Tipo: float64              | 

In [7]:
df_1=df.copy()

In [8]:
# Renombrar y reordenar columnas
df_1 = df_1.rename(columns={'target_m': 'target'})
df_1 = df_1[['target'] + [c for c in df_1.columns if c != 'target']]

In [9]:
df_1 = df_1[['target'] + [c for c in df_1.columns if c != 'target']]
df_1 = df_1.drop_duplicates(subset=['key_value', 'cod_mes'], keep='first')

In [10]:
import pandas as pd
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════
# 8.2 TRATAMIENTO DE VALORES FALTANTES — ESTRATEGIAS DIFERENCIADAS
# ═══════════════════════════════════════════════════════════════════════════

df_2 = df_1.copy()
df_pre = df_2  # alias — ambos apuntan al mismo objeto

# ── 1. VARIABLES TRANSACCIONALES → imputar con 0 ────────────────────────────
cols_transac = [c for c in df_pre.columns if c.startswith(("cnt_trx_", "imp_trx_", "max_trx_", "avg_trx_"))]
df_pre[cols_transac] = df_pre[cols_transac].fillna(0)

# ── 2. RATIOS FINANCIEROS → imputar con mediana ──────────────────────────────
cols_ratios = [c for c in df_pre.columns if c.startswith(("rat_", "ratio_", "share_", "gap_"))]
for col in cols_ratios:
    mediana = df_pre[col].median()
    df_pre[col] = df_pre[col].fillna(mediana)

# ── 3. VARIABLES DE VARIACIÓN (cnt_meses_sin*, flg_vrcn_*) → -1 ─────────────
cols_variacion = [c for c in df_pre.columns if c.startswith(("cnt_meses_sin", "flg_vrcn_"))]
df_pre[cols_variacion] = df_pre[cols_variacion].fillna(-1)

# ── 4. ANTIGÜEDAD → mediana de la población ──────────────────────────────────
if "num_antiguedad" in df_pre.columns:
    mediana_antig = df_pre["num_antiguedad"].median()
    df_pre["num_antiguedad"] = df_pre["num_antiguedad"].fillna(mediana_antig)

# ── 5. VARIABLES FLAG (flg_*) → 0 ────────────────────────────────────────────
cols_flag = [c for c in df_pre.columns if c.startswith("flg_") and c not in cols_variacion]
df_pre[cols_flag] = df_pre[cols_flag].fillna(0)

# ── 6. Columnas Int32 → convertir a int64 ────────────────────────────────────
int32_cols = df_pre.select_dtypes(include=["Int32"]).columns
df_pre[int32_cols] = df_pre[int32_cols].fillna(0).astype("int64")

# ── 7. Columnas boolean → rellenar NA con False ───────────────────────────────
bool_cols = df_pre.select_dtypes(include=["boolean"]).columns
df_pre[bool_cols] = df_pre[bool_cols].fillna(False)

# ── 8. VARIABLES CATEGÓRICAS RESTANTES → "SIN_INFO" ─────────────────────────
cols_cat = df_pre.select_dtypes(include=["object", "string"]).columns.tolist()
df_pre[cols_cat] = df_pre[cols_cat].fillna("SIN_INFO")

# ── 9. RESTO DE NUMÉRICAS → mediana ──────────────────────────────────────────
cols_num_restantes = df_pre.select_dtypes(include=[np.number]).columns
for col in cols_num_restantes:
    if df_pre[col].isnull().sum() > 0:
        df_pre[col] = df_pre[col].fillna(df_pre[col].median())

# ── Validación ───────────────────────────────────────────────────────────────
nulls_post = df_pre.isnull().sum()
nulls_post = nulls_post[nulls_post > 0]

print("=" * 65)
print("8.2  IMPUTACIÓN COMPLETADA")
print("=" * 65)
print(f"  Shape: {df_pre.shape}")
if nulls_post.empty:
    print("  ✓ 0 valores faltantes después de imputación")
else:
    print(f"  ⚠ Columnas con nulos pendientes ({len(nulls_post)}):")
    print(nulls_post.to_string())
print(f"\n  Grupos imputados:")
print(f"    Transaccionales  : {len(cols_transac):>3} cols → 0")
print(f"    Ratios           : {len(cols_ratios):>3} cols → mediana")
print(f"    Variación        : {len(cols_variacion):>3} cols → -1")
print(f"    Flags            : {len(cols_flag):>3} cols → 0")
print(f"    Categóricas      : {len(cols_cat):>3} cols → 'SIN_INFO'")


8.2  IMPUTACIÓN COMPLETADA
  Shape: (534318, 66)
  ⚠ Columnas con nulos pendientes (2):
desc_nivel_rsg_lsb_tot    534318
gap_riesgo_pep_lsb        534318

  Grupos imputados:
    Transaccionales  :   8 cols → 0
    Ratios           :  14 cols → mediana
    Variación        :   4 cols → -1
    Flags            :   9 cols → 0
    Categóricas      :  13 cols → 'SIN_INFO'


## 8.2 Tratamiento de Valores Faltantes (Estrategias Diferenciadas)

### Variables Numéricas — Imputación Diferenciada

| Tipo de Variable | Estrategia de Imputación | Justificación |
|---|---|---|
| Transaccionales (`cnt_trx_*`, `imp_trx_*`) | `0` (Ausencia = No hay operación) | Interpretación: sin actividad |
| Ratios Financieros (`ratio_*`, `rat_*`) | Valor mediano por segmento | Evitar distorsión de ratios |
| Variables de Variación (`cnt_meses_sin*`, `flg_vrcn_*`) | `-1` o mediana de cohort | Indica cambio/reducción |
| Antigüedad (`num_antiguedad`) | Mediana de la población (últimas 12 meses) | Cliente reciente o sin registro |

### Variables Categóricas — Imputación

| Tipo de Variable | Estrategia | Justificación |
|---|---|---|
| Variables de Riesgo (`cod_nivel_riesgo`) | `"BAJO"` | Defecto: categoría menor |
| Variables Flag (0/1) | `0` (Ausencia = No) | Interpretación conservadora |
| Resto de Categóricas | `"SIN_INFO"` | Marca explícita de missing |

### Variables de Fecha — Transformación

Las fechas **no son imputadas**. Se convierten en variables derivadas:
- Diferencia de días desde eventos relevantes
- Antigüedad en meses desde fecha de referencia
- Indicadores de presencia/ausencia de fecha

## 8.4 Codificación de Variables Categóricas

**Método:** Label Encoding (Mapeo Ordenado)

**Proceso:**
1. Identificación de variables categóricas de alta cardinalidad
2. Reagrupamiento en categorías de riesgo: Bajo, Medio, Alto, Nulo
3. Asignación de valores numéricos ordenados:
   - `Nulo`: 0
   - `Bajo`: 1
   - `Medio`: 2
   - `Alto`: 3

**Criterio de Agrupamiento:**
- Cantidad de casos positivos por categoría
- Efectividad (tasa de positividad) de cada categoría
- Relevancia de negocio

**Ejemplo `cod_ubigeo_cd`:**
- Original: 100+ ciudades
- Reagrupado: Bajo Riesgo (provincias) → Medio Riesgo (ciudades medianas) → Alto Riesgo (Lima metropolitana)

In [11]:
import pandas as pd
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════
# 8.4 CODIFICACIÓN DE VARIABLES CATEGÓRICAS (Label Encoding Ordenado)
# Escala: Nulo=0, Bajo=1, Medio=2, Alto=3
# ═══════════════════════════════════════════════════════════════════════════

TARGET_COL = "target"

# Identificar variables categóricas (object / string)
cols_cat_encode = [c for c in df_pre.select_dtypes(include=["object", "string"]).columns
                   if c != TARGET_COL and c not in ("key_value", "cod_cli", "tipo_alerta_n2",
                                                      "trx_riesgo_cliente", "desc_provincia",
                                                      "desc_departamento")]

print("=" * 65)
print("8.4  CODIFICACIÓN DE VARIABLES CATEGÓRICAS")
print("=" * 65)
print(f"  Variables a codificar: {len(cols_cat_encode)}\n")

ENCODING_MAPS = {}

for col in cols_cat_encode:
    # Calcular tasa de positividad por categoría
    tasa = (
        df_pre.groupby(col)[TARGET_COL]
        .agg(count="count", pos_rate="mean")
        .reset_index()
        .sort_values("pos_rate")
    )

    n_cats = len(tasa)
    # Terciles para asignar Bajo / Medio / Alto
    if n_cats >= 3:
        tercil1 = int(np.floor(n_cats / 3))
        tercil2 = int(np.floor(2 * n_cats / 3))
        bajo  = tasa.iloc[:tercil1][col].tolist()
        medio = tasa.iloc[tercil1:tercil2][col].tolist()
        alto  = tasa.iloc[tercil2:][col].tolist()
    elif n_cats == 2:
        bajo  = [tasa.iloc[0][col]]
        medio = []
        alto  = [tasa.iloc[1][col]]
    else:
        bajo  = tasa[col].tolist()
        medio = []
        alto  = []

    mapa = {"SIN_INFO": 0}
    for v in bajo:
        mapa[v] = 1
    for v in medio:
        mapa[v] = 2
    for v in alto:
        mapa[v] = 3

    ENCODING_MAPS[col] = mapa
    df_pre[col] = df_pre[col].map(mapa).fillna(0).astype(int)
    print(f"  ✓ {col:<40} | categorías={n_cats} | bajo={len(bajo)}, medio={len(medio)}, alto={len(alto)}")

print(f"\n  ✅ Codificación completada para {len(cols_cat_encode)} variables")


8.4  CODIFICACIÓN DE VARIABLES CATEGÓRICAS
  Variables a codificar: 7

  ✓ codmes_lag1                              | categorías=3 | bajo=1, medio=1, alto=1
  ✓ fec_constitucion                         | categorías=9888 | bajo=3296, medio=3296, alto=3296
  ✓ cod_ubigeo_cd                            | categorías=1270 | bajo=423, medio=423, alto=424
  ✓ cod_sectorista_id                        | categorías=705 | bajo=235, medio=235, alto=235
  ✓ cod_ciiu_v4                              | categorías=407 | bajo=135, medio=136, alto=136
  ✓ mto_fact_declarado_sunat                 | categorías=39937 | bajo=13312, medio=13312, alto=13313
  ✓ flg_activo_pep                           | categorías=2 | bajo=1, medio=0, alto=1

  ✅ Codificación completada para 7 variables


## 8.5 Cuantificación de Missing

**Umbral de Eliminación:** Variables con `>20%` missing se evalúan para exclusión (salvo variables críticas de negocio).

```sql
SELECT 
  variable_name,
  COUNT(*) AS total_registros,
  SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS missing_count,
  ROUND(100 * SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_missing
FROM dataset_extraido
GROUP BY variable_name
ORDER BY pct_missing DESC
```

In [12]:
import pandas as pd
import os

# ═══════════════════════════════════════════════════════════════════════════
# 8.5 CUANTIFICACIÓN DE MISSING (Pre y Post imputación)
# ═══════════════════════════════════════════════════════════════════════════

OUTPUT_DIR_PRE = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarollo\preprocesamiento"
os.makedirs(OUTPUT_DIR_PRE, exist_ok=True)

UMBRAL_MISSING = 20.0  # % — variables por encima se evalúan para exclusión

# Missing PRE-imputación (sobre df_2)
missing_pre = (df_2.isnull().sum() / len(df_2) * 100).round(2).sort_values(ascending=False)

# Missing POST-imputación (sobre df_pre)
missing_post = (df_pre.isnull().sum() / len(df_pre) * 100).round(2).sort_values(ascending=False)

df_missing = pd.DataFrame({
    "variable"       : missing_pre.index,
    "total_registros": len(df_2),
    "missing_pre_pct": missing_pre.values,
    "missing_post_pct": [missing_post.get(v, 0) for v in missing_pre.index],
}).reset_index(drop=True)

df_missing["evaluar_exclusion"] = df_missing["missing_pre_pct"] > UMBRAL_MISSING

# Mostrar variables sobre umbral
sobre_umbral = df_missing[df_missing["evaluar_exclusion"]]
print("=" * 65)
print("8.5  CUANTIFICACIÓN DE MISSING")
print("=" * 65)
print(f"  Umbral evaluación: >{UMBRAL_MISSING}%")
print(f"  Variables sobre umbral PRE-imputación: {len(sobre_umbral)}\n")

if not sobre_umbral.empty:
    display(sobre_umbral[["variable", "missing_pre_pct", "missing_post_pct"]].style
            .background_gradient(subset=["missing_pre_pct"], cmap="Reds")
            .format({"missing_pre_pct": "{:.1f}%", "missing_post_pct": "{:.1f}%"}))
else:
    print("  ✓ Ninguna variable supera el umbral de missing")

# Guardar reporte
fp_missing = os.path.join(OUTPUT_DIR_PRE, "01_Missing_PreImputacion.csv")
df_missing.to_csv(fp_missing, index=False)
print(f"\n  ✓ Reporte guardado: {fp_missing}")


8.5  CUANTIFICACIÓN DE MISSING
  Umbral evaluación: >20.0%
  Variables sobre umbral PRE-imputación: 2



,variable,missing_pre_pct,missing_post_pct
0,gap_riesgo_pep_lsb,100.0%,100.0%
1,desc_nivel_rsg_lsb_tot,100.0%,100.0%



  ✓ Reporte guardado: c:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarollo\preprocesamiento\01_Missing_PreImputacion.csv


## 8.6 Controles Posteriores al Preprocesamiento

**Validaciones ejecutadas:**
1. **Ausencia de Nulls** — todas las variables deben tener 0 valores faltantes
2. **Validación de Tipos** — numéricas deben ser `float` o `int`
3. **Detección de Outliers** — percentiles 1% y 99%; decisión: mantener o winsorizar (cap p99)
4. **Distribución de Variables** — verificación de simetría y anomalías

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

# ═══════════════════════════════════════════════════════════════════════════
# 8.6 CONTROLES POSTERIORES AL PREPROCESAMIENTO
# ═══════════════════════════════════════════════════════════════════════════

OUTPUT_DIR_PRE = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarollo\preprocesamiento"
os.makedirs(OUTPUT_DIR_PRE, exist_ok=True)

cols_num = df_pre.select_dtypes(include=[np.number]).columns.tolist()
if "target" in cols_num:
    cols_num.remove("target")

print("=" * 65)
print("8.6  CONTROLES POSTERIORES AL PREPROCESAMIENTO")
print("=" * 65)

# ── Control 1: Ausencia de Nulls ─────────────────────────────────────────────
nulls = df_pre.isnull().sum()
nulls = nulls[nulls > 0]
print(f"\n1️⃣  Nulls post-imputación: {'✓ 0 nulos' if nulls.empty else f'⚠ {len(nulls)} columnas con nulos'}")
if not nulls.empty:
    print(nulls.to_string())

# ── Control 2: Validación de tipos ───────────────────────────────────────────
tipos_incorrectos = df_pre[cols_num].dtypes[~df_pre[cols_num].dtypes.isin(["float64", "int64", "int32", "float32"])]
print(f"\n2️⃣  Tipos de datos numéricos: {'✓ Todos float/int' if tipos_incorrectos.empty else f'⚠ Tipos no esperados: {tipos_incorrectos.to_dict()}'}")

# ── Control 3: Detección de Outliers (p1 y p99) ──────────────────────────────
print("\n3️⃣  Detección de Outliers (percentiles 1% y 99%):")
rows_outliers = []
for col in cols_num:
    p1  = df_pre[col].quantile(0.01)
    p99 = df_pre[col].quantile(0.99)
    pct_outliers = ((df_pre[col] < p1) | (df_pre[col] > p99)).mean() * 100
    rows_outliers.append({"variable": col, "p1": p1, "p99": p99, "pct_outliers": round(pct_outliers, 2)})

df_outliers = pd.DataFrame(rows_outliers).sort_values("pct_outliers", ascending=False)

# Winsorización automática en variables con outliers > 5%
cols_winsorizadas = []
for _, row in df_outliers[df_outliers["pct_outliers"] > 5].iterrows():
    col = row["variable"]
    df_pre[col] = df_pre[col].clip(lower=row["p1"], upper=row["p99"])
    cols_winsorizadas.append(col)

print(f"  Variables winsorizadas (outliers >5%): {len(cols_winsorizadas)}")
for c in cols_winsorizadas:
    print(f"    → {c}")

fp_outliers = os.path.join(OUTPUT_DIR_PRE, "03_Outliers_Detectados.csv")
df_outliers.to_csv(fp_outliers, index=False)
print(f"  ✓ Reporte guardado: {fp_outliers}")

# ── Control 4: Estadísticos post-imputación ───────────────────────────────────
print("\n4️⃣  Estadísticos post-imputación:")
df_stats_post = df_pre[cols_num].describe().T
df_stats_post["skewness"] = df_pre[cols_num].skew().round(3)
df_stats_post["kurtosis"] = df_pre[cols_num].kurt().round(3)

fp_stats = os.path.join(OUTPUT_DIR_PRE, "02_Estadisticos_PostImputacion.csv")
df_stats_post.to_csv(fp_stats)
print(f"  ✓ Reporte guardado: {fp_stats}")

display(df_stats_post[["mean", "std", "min", "50%", "max", "skewness", "kurtosis"]]
        .style.background_gradient(subset=["skewness"], cmap="RdYlGn_r")
        .format("{:.3f}"))

print(f"""
╔══════════════════════════════════════════════════════╗
║          CONTROLES COMPLETADOS ✅                    ║
╠══════════════════════════════════════════════════════╣
║  Nulls post-imputación : {'0' if nulls.empty else str(len(nulls)):<35}║
║  Variables winsorizadas: {len(cols_winsorizadas):<35}║
║  Reportes generados:                                 ║
║    01_Missing_PreImputacion.csv                      ║
║    02_Estadisticos_PostImputacion.csv                ║
║    03_Outliers_Detectados.csv                        ║
╚══════════════════════════════════════════════════════╝
""")


8.6  CONTROLES POSTERIORES AL PREPROCESAMIENTO

1️⃣  Nulls post-imputación: ⚠ 2 columnas con nulos
desc_nivel_rsg_lsb_tot    534318
gap_riesgo_pep_lsb        534318

2️⃣  Tipos de datos numéricos: ⚠ Tipos no esperados: {'codmes_lag1': dtype('int64'), 'cod_mes': dtype('int64'), 'fec_constitucion': dtype('int64'), 'cnt_trx_cargostot_3m': dtype('int64'), 'num_edad_constitucion': dtype('int64'), 'num_antiguedad': dtype('int64'), 'cnt_meses_siningresos_12m': dtype('int64'), 'cnt_meses_sinegresos_12m': dtype('int64'), 'cod_ubigeo_cd': dtype('int64'), 'cod_sectorista_id': dtype('int64'), 'cod_ciiu_v4': dtype('int64'), 'flg_casos_hist': dtype('int64'), 'flg_vrcn_abonos_5m_1m': dtype('int64'), 'flg_vrcn_efe_cargos_5m_1m': dtype('int64'), 'cnt_ro_debajo_umbral': dtype('int64'), 'mto_fact_declarado_sunat': dtype('int64'), 'flg_al_ext_12m': dtype('int64'), 'flg_del_ext_12m': dtype('int64'), 'cnt_trx_sinenv_alext_12m': dtype('int64'), 'cnt_trx_al_ext_1000_12m': dtype('int64'), 'flg_pep': dtype('int

,mean,std,min,50%,max,skewness,kurtosis
codmes_lag1,1.988,0.815,1.000,2.000,3.000,0.022,-1.495
cod_mes,202606.012,0.815,202605.000,202606.000,202607.000,-0.022,-1.495
fec_constitucion,2.786,0.480,1.000,3.000,3.000,-2.189,4.052
mto_pas_soles,72237.428,4925303.949,0.000,241.185,2042446186.460,293.840,107632.350
imp_trx_abonosefect_6m,29638.673,424659.210,0.000,0.000,121530768.945,92.301,17332.373
imp_trx_cargosefe_6m,28550.562,352345.871,0.000,0.000,63939801.300,61.189,5955.712
avg_trx_cargostot_3m,0.546,1.959,0.000,0.000,141.333,9.192,191.125
max_trx_abonos_3m,221421.594,9194065.885,0.000,4221.000,2194772535.934,146.102,27510.943
cnt_trx_cargostot_3m,49.554,476.833,0.000,10.000,115547.000,136.347,24900.818
cnt_trx_abonospromtot_3m,37.883,1262.832,0.000,1.000,511889.000,342.698,133431.495



╔══════════════════════════════════════════════════════╗
║          CONTROLES COMPLETADOS ✅                    ║
╠══════════════════════════════════════════════════════╣
║  Nulls post-imputación : 2                                  ║
║  Variables winsorizadas: 0                                  ║
║  Reportes generados:                                 ║
║    01_Missing_PreImputacion.csv                      ║
║    02_Estadisticos_PostImputacion.csv                ║
║    03_Outliers_Detectados.csv                        ║
╚══════════════════════════════════════════════════════╝



## 8.7 Evidencia

Consultar los siguientes artefactos generados por este notebook:

| Artefacto | Descripción |
|---|---|
| `01_Missing_PreImputacion.csv` | % de nulos por variable antes de imputación |
| `02_Estadisticos_PostImputacion.csv` | Estadísticos descriptivos post-imputación |
| `03_Outliers_Detectados.csv` | Variables con outliers detectados (p1/p99) |

**Scripts de referencia:**
- Script: `1.preprocesamiento.py`
- Notebook análisis: `03_analisis_preprocesamiento.ipynb`

📁 Ruta de reportes: `PLAFT/PJ/Minorista/Final/Desarollo/preprocesamiento/`

In [14]:
df_2[['cod_mes', 'target']].value_counts()

cod_mes  target
202607   0         180766
202606   0         178985
202605   0         174251
         1            224
202606   1             78
202607   1             14
Name: count, dtype: int64

In [15]:
import pandas as pd
from sklearn.utils import resample

# ===========================
# 1️⃣ Separar antiguos y recientes
# ===========================
df_antiguos = df_2[df_2['cod_mes'] <= 202507].copy()
df_recientes = df_2[df_2['cod_mes'].between(202508, 202607)].copy()
#df_recientes = df_2[df_2['cod_mes'] >= 202508].copy()  # Test completo

# ===========================
# 2️⃣ Separar clases en antiguos
# ===========================
df_antiguos_con_alerta = df_antiguos[df_antiguos['tipo_alerta_n2'] != "0"]  # Con alerta
df_antiguos_sin_alerta = df_antiguos[df_antiguos['tipo_alerta_n2'] == "0"]  # Sin alerta

# ===========================
# 3️⃣ Filtrar clases target == 1 (minoritarios)
# ===========================
df_antiguos_1 = df_antiguos_con_alerta[df_antiguos_con_alerta['target'] == 1]  # Alerta y target == 1
df_antiguos_0 = df_antiguos_con_alerta[df_antiguos_con_alerta['target'] == 0]  # Alerta y target == 0

# ===========================
# 4️⃣ Balanceo de clases en TRAIN
# ===========================
n_pos = len(df_antiguos_1)  # Cantidad de clases 1 (minoritarias)
n_neg_deseado = int(n_pos * (99.5 / 0.5))  # Queremos 99.5% de clase 0

# Ajustar clase 0 (target == 0)
if n_neg_deseado <= len(df_antiguos_0):
    df_antiguos_0_bal = resample(df_antiguos_0, replace=False, n_samples=n_neg_deseado, random_state=42)
else:
    df_antiguos_0_bal = resample(df_antiguos_0, replace=True, n_samples=n_neg_deseado, random_state=42)

# ===========================
# 5️⃣ Combinar con clases 1 (target == 1)
# ===========================
df_antiguos_1_bal = df_antiguos_1  # Mantener todas las clases 1 (no se ajustan, ya que se mantiene su porcentaje)

# Combinar 0 (balanceado) y 1 (original) para el dataset de entrenamiento
df_antiguos_balanceados = pd.concat([df_antiguos_0_bal, df_antiguos_1_bal], axis=0)

# ===========================
# 6️⃣ Tomamos el 0.5% de los SIN alerta
# ===========================
# Para tener el 99.5% de clase 0 y 0.5% de clase 1 en el TRAIN
porcentaje_sin_alerta = 0.005  # 0.5% de los registros sin alerta

# Tomamos el 0.5% de los casos sin alerta (sin alertas)
df_sin_alerta_sample = df_antiguos_sin_alerta.sample(frac=porcentaje_sin_alerta, random_state=42)

# ===========================
# 7️⃣ Concatenar alertas + 0.5% de sin alerta
# ===========================
df_train = pd.concat([df_antiguos_balanceados, df_sin_alerta_sample], axis=0)

# Barajamos los datos para evitar sesgo de orden
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

# ===========================
# 8️⃣ Mantener df_test intacto
# ===========================
# Filtrar df_test para asegurarnos de que tiene solo los registros del test (cod_mes >= 202508)
df_test=df_2[df_2['cod_mes'].between(202508, 202607)].copy()
#df_test = df_3[df_3['cod_mes'] >= 202508].copy()

# ===========================
# 9️⃣ Combinar df_train y df_test para df_7
# ===========================
df_3 = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

# ===========================
# 10️⃣ Borrar columna 'tipo_alerta_n2' de df_7
# ===========================
#df_4 = df_4.drop(columns=['tipo_alerta_n2'], errors='ignore')

# ===========================
# 11️⃣ Verificar distribución final
# ===========================
print("Distribución final de clases en df_train (target):")
print(df_train['target'].value_counts(normalize=True))

print("\nDistribución final de tipo_alerta_n2 en df_train (Eliminada):")
print(df_train['tipo_alerta_n2'].value_counts())

print("\nDistribución final de clases en df_test (target):")
print(df_test['target'].value_counts(normalize=True))

print("\nDistribución final de clases en df_7 (target):")
print(df_3['target'].value_counts(normalize=True))

InvalidParameterError: The 'n_samples' parameter of resample must be an int in the range [1, inf) or None. Got 0 instead.

In [26]:
df_3=df_2 

In [27]:
# Eliminar columnas 'fec_constitucion' si existe
cols_a_eliminar = ["fec_constitucion"]
df_3 = df_3.drop(columns=cols_a_eliminar, errors="ignore")

In [28]:
df_3['tipo_alerta_n2'] = df_3['tipo_alerta_n2'].astype(str)

In [29]:
df_3["cnt_alerta_hist"] = (
    df_3["cnt_alerta_hist"]
    .astype("float64")
)

In [30]:
df_3["mto_fact_declarado_sunat"] = pd.to_numeric(df_3["mto_fact_declarado_sunat"], errors="coerce")

In [31]:
import pandas as pd

# ═══════════════════════════════════════════════════════════════════════════
# Pasos finales alineados con preprocessing_1.py antes de guardar df_3
# ═══════════════════════════════════════════════════════════════════════════

# Columnas de identificación/control (preprocessing_1.py::COLS_IDS) — se conservan
COLS_IDS = ["cod_mes", "key_value", "cod_cli", "tipo_alerta_n2", "trx_riesgo_cliente"]

# Eliminar 'codmes_lag1' (preprocessing_1.py::limpieza_inicial → COLS_DROP_ALWAYS)
df_3 = df_3.drop(columns=["codmes_lag1"], errors="ignore")

# Selección de columnas finales del modelo (preprocessing_1.py::seleccionar_columnas) + COLS_IDS
COLUMNS_FILE = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\Interbank\PLAFT\Desarrollo\Minorista_regulado\Train\selected_columns.csv"
columnas_modelo = pd.read_csv(COLUMNS_FILE, header=None)[0].tolist()
if "target" not in columnas_modelo:
    columnas_modelo = ["target"] + columnas_modelo
columnas_modelo += [c for c in COLS_IDS if c not in columnas_modelo]

faltantes = [c for c in columnas_modelo if c not in df_3.columns]
if faltantes:
    print(f"⚠ No encontradas en df_3: {faltantes}")

cols_ok = [c for c in columnas_modelo if c in df_3.columns]
df_3 = df_3[cols_ok]
print(f"✓ Selección de columnas completada: {df_3.shape[1]} de {len(columnas_modelo)} columnas esperadas")


✓ Selección de columnas completada: 38 de 38 columnas esperadas


In [32]:
df_3.head()

,target,cnt_trx_cargostot_3m,mto_pas_soles,rat_pastot_x_ingtot_6m,cnt_trx_abonospromtot_3m,imp_trx_abonosefect_6m,num_antiguedad,imp_trx_cargosefe_6m,ratio_cargos_1m_vs_6m,cnt_meses_sinegresos_12m,...,cnt_noticias,share_cp_ingresos,mto_fact_declarado_sunat,cod_ubigeo_cd,cod_sectorista_id,cod_mes,key_value,cod_cli,tipo_alerta_n2,trx_riesgo_cliente
0,0,0,0.00,0.02,0.00,0.00,1,0.00,6034.99,12,...,0,0.00,2,3,2,202606,5FC20C5D0C19B187AAAD242986898510AD5B03D2A75DEA...,0020981707,SIN_INFO,SIN_INFO
1,0,0,0.00,0.00,0.00,0.00,9,0.00,6034.99,8,...,3,0.00,2,3,3,202607,0682DA8030BDDC346E9F14826A323DAEB1B6B313B5F784...,0014834336,SIN_INFO,SIN_INFO
2,0,338,20683.73,0.02,97.67,13751.80,13,14800.00,2467163.89,8,...,3,0.06,3,3,1,202605,778D6D69E6D7BE755FD9022085F14E9FC651B892B93A04...,0013076463,SIN_INFO,SIN_INFO
3,0,1,25.00,0.25,0.00,0.00,4,0.00,6034.99,10,...,3,0.00,3,3,3,202607,6EB7D03B127910B5FBDA93481A055B12F128CE7025C476...,0018760243,SIN_INFO,SIN_INFO
4,0,5,0.00,0.00,0.33,0.00,0,0.00,6034.99,5,...,0,0.00,3,3,3,202606,0331318F473AAE318596BB84B95B0C194B786C96E0A256...,0021907687,SIN_INFO,SIN_INFO


In [33]:
df_3.to_parquet(
    's3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/DATA_INFERENCIA/data_pn_total_expandido_new_infere_0726.parquet',
    index=False
)

In [ ]:
import json
import pickle
import os
import boto3

# ═══════════════════════════════════════════════════════════════════════════
# GUARDAR ARTEFACTOS DE PREPROCESAMIENTO PARA INFERENCIA
# ─────────────────────────────────────────────────────────────────────────
# Se guardan dos artefactos generados durante el entrenamiento:
#   1. ENCODING_MAPS  → mapas de Label Encoding (sección 8.4)
#   2. percentil_limits → límites de winsorización p1/p99 (sección 8.6)
#
# Estos artefactos deben aplicarse TAL CUAL en inferencia para evitar
# data leakage (no recalcular sobre datos nuevos).
# ═══════════════════════════════════════════════════════════════════════════

S3_BUCKET = "ibk-discovery-comercial-us-east-1-654654352211-data"
S3_PREFIX = "discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/MODEL/artifacts_v2"
LOCAL_DIR  = r"c:\Users\b46637\OneDrive - Interbank\PLAFT\PJ\Minorista\Final\Desarollo\preprocesamiento"
os.makedirs(LOCAL_DIR, exist_ok=True)

# ── 1. ENCODING_MAPS → JSON (serializable, legible) ─────────────────────────
fp_enc_local = os.path.join(LOCAL_DIR, "encoding_maps.json")
with open(fp_enc_local, "w", encoding="utf-8") as f:
    json.dump(ENCODING_MAPS, f, ensure_ascii=False, indent=2)
print(f"✓ ENCODING_MAPS guardado localmente: {fp_enc_local}")
print(f"  Variables codificadas: {len(ENCODING_MAPS)}")

# ── 2. PERCENTIL_LIMITS para winsorización → JSON ────────────────────────────
# Construir desde df_outliers generado en sección 8.6
percentil_limits = {}
for _, row in df_outliers[df_outliers["pct_outliers"] > 5].iterrows():
    percentil_limits[row["variable"]] = (float(row["p1"]), float(row["p99"]))

fp_wins_local = os.path.join(LOCAL_DIR, "percentil_limits.json")
with open(fp_wins_local, "w", encoding="utf-8") as f:
    json.dump(percentil_limits, f, ensure_ascii=False, indent=2)
print(f"✓ PERCENTIL_LIMITS guardado localmente: {fp_wins_local}")
print(f"  Variables winsorizadas: {len(percentil_limits)}")

# ── 3. Subir ambos artefactos a S3 ───────────────────────────────────────────
s3 = boto3.client("s3")

for local_file, s3_key_suffix in [
    (fp_enc_local,  "encoding_maps.json"),
    (fp_wins_local, "percentil_limits.json"),
]:
    s3_key = f"{S3_PREFIX}/{s3_key_suffix}"
    s3.upload_file(local_file, S3_BUCKET, s3_key)
    print(f"✓ Subido a s3://{S3_BUCKET}/{s3_key}")

print(f"""
╔══════════════════════════════════════════════════════════════╗
║  ARTEFACTOS DE PREPROCESAMIENTO GUARDADOS ✅                 ║
╠══════════════════════════════════════════════════════════════╣
║  encoding_maps.json    → {len(ENCODING_MAPS)} variables codificadas            ║
║  percentil_limits.json → {len(percentil_limits)} variables winsorizadas         ║
╠══════════════════════════════════════════════════════════════╣
║  S3: s3://{S3_BUCKET}/                   ║
║      {S3_PREFIX}/  ║
╚══════════════════════════════════════════════════════════════╝
""")


In [ ]:
df_3.shape

(1685224, 65)